In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
df = pd.read_csv("/kaggle/input/datasets/sonialikhan/heart-attack-analysis-and-prediction-dataset/heart.csv")
import pandas as pd

In [ ]:
import pandas as pd
import seaborn as sb
import matplotlib.pyplot as plt

# 1. Veriyi doğru fonksiyonla yüklüyoruz
df = pd.read_csv("/kaggle/input/datasets/sonialikhan/heart-attack-analysis-and-prediction-dataset/heart.csv")

# 2. Sadece sayısal sütunları seçiyoruz (Korelasyon için şart)
numeric_df = df.select_dtypes(include=['number'])

# 3. Grafik boyutunu ayarlıyoruz (Korelasyonlar rahat okunsun diye)
plt.figure(figsize=(10, 8))

# 4. Heatmap'i korelasyon matrisi üzerinden oluşturuyoruz
sb.heatmap(numeric_df.corr(), cmap="coolwarm", annot=True, fmt=".2f", linewidths=0.5)
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


# Grafik stilini ayarlayalım
sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 6))

# Target/Output kırılımlı Göğüs Ağrısı Dağılımı
# hue='output' (veya veri setindeki adıyla 'target') çubukları kriz durumuna göre böler
ax = sns.countplot(
    data=df, 
    x='cp', 
    hue='output', 
    palette='coolwarm', 
    alpha=0.9
)

# Grafik başlıkları ve etiketleri
plt.title('Göğüs Ağrısı Tiplerine Göre Kalp Krizi Riski Dağılımı', fontsize=14, pad=15)
plt.xlabel('Göğüs Ağrısı Tipi (cp)', fontsize=12)
plt.ylabel('Hasta Sayısı', fontsize=12)

# X eksenindeki değerlerin isimlerini netleştirelim
# Senin paylaştığın yeni sıralamaya göre güncellendi:
plt.xticks([0, 1, 2, 3], labels=[
    'Value 0\n(Typical Angina)', 
    'Value 1\n(Atypical Angina)', 
    'Value 2\n(Non-anginal Pain)', 
    'Value 3\n(Asymptomatic)'
])

# Sağ üstteki lejantı (bilgilendirme kutusunu) düzenleyelim
plt.legend(title='Kalp Krizi Riski', labels=['0 (Düşük Risk)', '1 (Yüksek Risk)'])

# Barların üzerine tam sayı değerlerini yazdırma (Kaggle'da görsel olarak çok işe yarar)
for container in ax.containers:
    ax.bar_label(container, fontsize=10, weight='bold')

# Kaggle notebook hücresinde grafiği göstermek için
plt.show()

In [ ]:
# 2. CP Tiplerine Göre Yaş ve Maksimum Kalp Hızı İlişkisi (Scatter)
g = sns.relplot(
    data=df,
    x='age',
    y='thalachh',
    col='cp',           
    hue='output',       
    kind='scatter',
    palette='coolwarm', 
    alpha=0.8,
    s=70,               
    col_wrap=2          
)

# Başlıkları düzenleme
g.fig.suptitle('Göğüs Ağrısı Tiplerine Göre Yaş ve Maksimum Kalp Hızı İlişkisi', fontsize=14, y=1.05)
g.set_titles("Göğüs Ağrısı Tipi: {col_name}")
g.set_axis_labels("Yaş (age)", "Maksimum Kalp Hızı (thalachh)")

plt.show()

# Model Fitting

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import lightgbm as lgb
from sklearn.metrics import accuracy_score, f1_score

# 1. Özellikleri (X) ve Hedef Değişkeni (y) Ayırma
# Veri setindeki sütun adı 'output' veya 'target' hangisiyse ona göre güncelle
X = df.drop(columns=['output']) 
y = df['output']

# 2. Train ve Test Setlerine Bölme (%80 Eğitim, %20 Test)
# Sınıf dengesini korumak için stratify=y kullanıyoruz
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(xl_train_shape := f"Eğitim seti boyutu: {X_train.shape}")
print(xl_test_shape := f"Test seti boyutu: {X_test.shape}")

In [ ]:
!pip install optuna

In [ ]:
import optuna
import lightgbm as lgb
from sklearn.metrics import f1_score
import warnings
warnings.filterwarnings('ignore') # Kalabalık logları gizlemek için
optuna.logging.set_verbosity(optuna.logging.CRITICAL)
def objective(trial):
    # 1. Optuna'nın deneyeceği hiperparametre aralıklarını tanımlıyoruz
    param = {
        'objective': 'binary',
        'metric': 'binary_logloss',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        
        # Optuna bu aralıklardan akıllı seçimler yapacak:
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 10, 100),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 40),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'n_estimators': trial.suggest_int('n_estimators', 50, 300)
    }
    
    # 2. Modeli bu denemedeki (trial) parametrelerle eğitiyoruz
    model = lgb.LGBMClassifier(**param)
    model.fit(X_train, y_train)
    
    # 3. Test seti üzerinde tahmin yapıyoruz
    preds = model.predict(X_test)
    
    # 4. Başarı kriteri olarak F1-Score seçiyoruz (Sinsi semptomları kaçırmamak adına dengeli bir metriktir)
    score = f1_score(y_test, preds)
    
    # Optuna bu skoru yukarı taşımaya çalışacak
    return score

In [ ]:
# Bir çalışma (study) oluşturuyoruz. Hedefimiz F1 skoru EN YÜKSEĞE (maximize) çıkarmak.
study = optuna.create_study(direction='maximize')

# Optimizasyonu başlatıyoruz (50 deneme yapacak)
study.optimize(objective, n_trials=50)

print("\n--- Optimizasyon Tamamlandı! ---")
print("En İyi F1 Skoru:", study.best_value)
print("En İyi Parametreler:", study.best_params)

In [ ]:
!pip install h2o

In [ ]:
import h2o
from h2o.estimators import H2OGradientBoostingEstimator

# H2O'yu başlatıyoruz (Arkada yerel bir Java sunucusu açar)
h2o.init()

In [ ]:

train_h2o = h2o.H2OFrame(pd.concat([X_train, y_train], axis=1))
test_h2o = h2o.H2OFrame(pd.concat([X_test, y_test], axis=1))

# H2O'da hedef değişkenin kategorik (sınıflandırma) olduğunu belirtmek şarttır:
train_h2o['output'] = train_h2o['output'].asfactor()
test_h2o['output'] = test_h2o['output'].asfactor()

# Özellik sütunlarını ve hedef sütunu tanımlayalım
features = X_train.columns.tolist()
target = 'output'

# 2. H2O GBM (Gradient Boosting) Modelini eğitiyoruz
h2o_model = H2OGradientBoostingEstimator(
    seed=42,
    nfolds=5 # 5-katlı çapraz doğrulama ile sinsi vakaları daha iyi öğrensin
)
h2o_model.train(x=features, y=target, training_frame=train_h2o)

In [ ]:
# Test verisi üzerindeki model performansını alıyoruz
performance = h2o_model.model_performance(test_data=test_h2o)

print("=== 1. H2O GAINS AND LIFT TABLE (DİLİMLER) ===")
# İstediğin o kümülatif capture rate, lift ve yüzdelik dilimlerin tablosu:
display(performance.gains_lift().as_data_frame())

print("\n=== 2. H2O OPTIMAL THRESHOLDS & METRICS TABLE (EŞİKLER) ===")
# KS istatistiğinin, F1 skorunun maksimum olduğu tüm optimal eşik değerlerinin tablosu:
display(performance.thresholds_and_metric_scores().as_data_frame())